In [ ]:
###################################################################################################
# 參數設定
audio_file = "test_audio.mp3"  # 你的本地錄音檔案名稱(含副檔名)
lang = "zh"                 # 辨識語言（中文為'zh'，英文'en'，日文'ja'，韓文'ko'，自動偵測用None）
output_format = "txt"       # 輸出格式 'txt' 或 'srt'
model_type = "turbo"        # 選擇辨識模型 'small', 'medium', 'large', 'turbo'(新)
output_path = "."           # 輸出資料夾，預設為目前目錄

overwrite = False           # 是否覆蓋已存在的辨識結果 (True or False)
verbose = False             # 是否即時顯示語音辨識結果 (True or False)

####################################################################################################

import whisper
import os
import torch

# 檢查檔案是否存在
if not os.path.exists(audio_file):
    print(f"找不到音訊檔案: {audio_file}")
    exit(1)

# 載入模型
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = whisper.load_model("small", device=DEVICE)

if DEVICE == "cuda":
    print("使用 cuda 加速")
else:
    print("無可用硬體加速，使用 CPU 執行")

# 設定語言參數
if lang.lower() == "自動判斷":
    language = None
elif lang.lower() in ["chinese", "zh"]:
    language = "zh"
elif lang.lower() in ["english", "en"]:
    language = "en"
elif lang.lower() in ["japanese", "ja"]:
    language = "ja"
elif lang.lower() in ["korean", "ko"]:
    language = "ko"
else:
    language = None

# 辨識音訊
result = model.transcribe(audio_file, language=language, verbose=verbose)

# 設定輸出檔名
base_name = os.path.splitext(os.path.basename(audio_file))[0]
output_file = os.path.join(output_path, f"{base_name}.{output_format}")

# 檢查是否需要覆蓋
count = 0
while os.path.exists(output_file) and not overwrite:
    count += 1
    output_file = os.path.join(output_path, f"{base_name}_{count}.{output_format}")

if output_format == "txt":
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(result["text"])
elif output_format == "srt":
    from whisper.utils import get_writer
    srt_writer = get_writer("srt", output_path)
    srt_writer(result, base_name)
print(f"辨識完成，結果已存至 {output_file}")